# 📊 Data Profiling — Team Outliers

**Mục tiêu**: Kiểm tra tổng quan 15 file dữ liệu, phát hiện giá trị thiếu (null), kiểu dữ liệu và các vấn đề bất thường.

In [ ]:
import polars as pl
import sys
from pathlib import Path

# Add src to path to import DataLoader
sys.path.append(str(Path("..").resolve()))
from src.data_loader import DataLoader

loader = DataLoader()
all_dfs = loader.load_all()

## 1. Kiểm tra số lượng dòng và cột

In [ ]:
summary = []
for name, df in all_dfs.items():
    summary.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Size (MB)": round(df.estimated_size() / (1024 * 1024), 2)
    })

pl.DataFrame(summary).sort("Rows", descending=True)

## 2. Kiểm tra giá trị Null

In [ ]:
def check_nulls(name, df):
    null_counts = df.null_count()
    null_pct = (null_counts / len(df) * 100)
    return null_pct

for name, df in all_dfs.items():
    nulls = check_nulls(name, df)
    if nulls.sum_horizontal()[0] > 0:
        print(f"\n⚠️ {name} has null values:")
        # Filter columns with nulls > 0
        for col in df.columns:
            pct = nulls[col][0]
            if pct > 0:
                print(f"  - {col}: {pct:.2f}%")

## 3. Kiểu dữ liệu (Data Types)

In [ ]:
for name, df in all_dfs.items():
    print(f"\n--- {name} ---")
    print(df.schema)

## 4. Phân bố và thống kê mô tả (Summary Statistics)

In [ ]:
for name, df in all_dfs.items():
    print(f"\n--- {name} ---")
    num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]
    if num_cols:
        display(df.select(num_cols).describe())
    else:
        print("No numerical columns")

## 5. Phát hiện Outlier (Outlier Detection)

In [ ]:
def check_outliers_iqr(df, col):
    s = df.get_column(col).drop_nulls()
    if len(s) == 0:
        return 0, 0, 0
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    if q1 is None or q3 is None:
        return 0, 0, 0
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df.filter((pl.col(col) < lower_bound) | (pl.col(col) > upper_bound))
    return len(outliers), lower_bound, upper_bound

for name, df in all_dfs.items():
    num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]
    outlier_summary = []
    for col in num_cols:
        try:
            n_outliers, lb, ub = check_outliers_iqr(df, col)
            if n_outliers > 0:
                outlier_summary.append({
                    "Column": col,
                    "Outliers": n_outliers,
                    "%": round(n_outliers / len(df) * 100, 2)
                })
        except Exception as e:
            pass
    if outlier_summary:
        print(f"\n--- {name} Outliers ---")
        display(pl.DataFrame(outlier_summary).sort("Outliers", descending=True))
